# Week 8, Day 5: Backpropagation: the chain rule through a network

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w008/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1986)
theta = rng.uniform(-np.pi, np.pi, 200)                       # motion direction on each trial (radians)
X = np.column_stack([np.cos(theta), np.sin(theta)])           # direction as a unit vector
y = 10 + 30 * np.exp(2 * (np.cos(theta - 0.8) - 1)) + 2 * rng.normal(size=200)   # a von Mises tuning curve, Hz
params = {"W1": rng.normal(size=(2, 16)) * 0.8, "b1": np.zeros(16), "w2": rng.normal(size=16) * 0.5, "b2": np.array(0.0)}

def forward(p, X):
    Z = X @ p["W1"] + p["b1"]
    H = np.tanh(Z)
    return Z, H, H @ p["w2"] + p["b2"]

def loss(p, X, y):
    return 0.5 * np.mean((forward(p, X)[2] - y) ** 2)
print("initial loss", round(loss(params, X, y), 2))

### Exercise: Write the backward pass

Write `backward(p, X, y)` that returns a dictionary of gradients with the same keys and shapes as `params`, following the four steps of the derivation. The check that follows compares every entry with a central difference: the gradient check that every hand-written backward pass should pass.

In [ ]:
def backward(p, X, y):
    Z, H, yhat = forward(p, X)
    T = len(y)
    d_yhat = ...
    g = {}
    g["w2"] = ...
    g["b2"] = ...
    dH = ...
    dZ = ...
    g["W1"] = ...
    g["b1"] = ...
    return g

g = backward(params, X, y)
print({k: np.round(np.ravel(v)[:3], 4) for k, v in g.items()})

In [ ]:
import numpy as np
def num_grad(p, name, h=1e-6):
    g = np.zeros_like(np.asarray(p[name], dtype=float))
    it = np.nditer(g, flags=["multi_index"])
    for _ in it:
        i = it.multi_index
        q = {k: np.array(v, dtype=float, copy=True) for k, v in p.items()}
        q[name][i] += h; Lp = loss(q, X, y)
        q[name][i] -= 2 * h; Lm = loss(q, X, y)
        g[i] = (Lp - Lm) / (2 * h)
    return g

def test_shapes():
    "backward returns a gradient with the shape of every parameter"
    g = backward(params, X, y)
    for k in params:
        assert np.shape(g[k]) == np.shape(params[k]), k

def test_check():
    "every gradient matches a numerical gradient"
    g = backward(params, X, y)
    for k in params:
        assert np.allclose(g[k], num_grad(params, k), rtol=1e-4, atol=1e-6), k

test_shapes()
test_check()
print("All checks passed")

### Exercise: A tiny autodiff engine

Complete the `_backward` functions for multiplication and tanh. Each one receives the gradient arriving at the output (`out.grad`) and adds the correct share to each input's `.grad`, using the local derivative. Then differentiate a neuron, tanh(w x + b), with respect to w.

In [ ]:
import math

class Value:
    """A number that remembers how it was computed, so gradients can flow back through it."""
    def __init__(self, data, parents=()):
        self.data, self.grad, self._parents, self._backward = data, 0.0, parents, lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad += out.grad            # d(a + b)/da = 1
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            ...
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,))
        def _backward():
            ...
        out._backward = _backward
        return out

    def backward(self):
        order, seen = [], set()
        def visit(v):                        # topological order: every node after its inputs
            if v not in seen:
                seen.add(v)
                for p in v._parents:
                    visit(p)
                order.append(v)
        visit(self)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()

w, x, b = Value(0.8), Value(1.5), Value(-0.2)
r = (w * x + b).tanh()
r.backward()
print("r =", round(r.data, 4), " dr/dw =", round(w.grad, 4), " dr/dx =", round(x.grad, 4), " dr/db =", round(b.grad, 4))

In [ ]:
import numpy as np
def test_mul_add():
    "gradients of a*b + a are correct"
    a, b = Value(2.0), Value(-3.0)
    out = a * b + a
    out.backward()
    assert np.isclose(a.grad, -2.0) and np.isclose(b.grad, 2.0)

def test_tanh():
    "tanh backward uses 1 - tanh^2"
    x = Value(0.5)
    out = x.tanh()
    out.backward()
    assert np.isclose(x.grad, 1 - np.tanh(0.5) ** 2)

def test_neuron():
    "a whole neuron's weight gradient matches the chain rule"
    w, xv, bv = Value(0.8), Value(1.5), Value(-0.2)
    r = (w * xv + bv).tanh()
    r.backward()
    assert np.isclose(w.grad, (1 - np.tanh(0.8 * 1.5 - 0.2) ** 2) * 1.5)

test_mul_add()
test_tanh()
test_neuron()
print("All checks passed")

### Exercise: Learn a tuning curve

Train the network by gradient descent: for 2000 steps, compute the gradients with `backward` and move every parameter a step of size `lr` against its gradient. Record the loss each step in `losses`, then plot the learned tuning curve over the data.

In [ ]:
p = {k: np.array(v, dtype=float, copy=True) for k, v in params.items()}
lr = 0.05
losses = []
for step in range(2000):
    ...

grid = np.linspace(-np.pi, np.pi, 200)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].semilogy(losses); ax[0].set_xlabel("step"); ax[0].set_ylabel("loss")
ax[1].plot(theta, y, ".", ms=3, alpha=0.5); ax[1].plot(grid, forward(p, np.column_stack([np.cos(grid), np.sin(grid)]))[2], lw=2)
ax[1].set_xlabel("direction (rad)"); ax[1].set_ylabel("rate (Hz)"); plt.tight_layout(); plt.show()
print("final loss", round(losses[-1], 3))

In [ ]:
import numpy as np
def test_losses():
    "losses has one value per step and falls by more than a factor of 10"
    assert len(losses) == 2000
    assert losses[-1] < losses[0] / 10

def test_fit():
    "the trained network's error is close to the noise level (variance 4)"
    assert losses[-1] < 4.0

test_losses()
test_fit()
print("All checks passed")